# Token Analysis: Date Injection in the Embedding Pipeline

This notebook checks how many tokens a date takes up, to estimate the impact of the date injection for rq3 on the overall embedding (max. 512 tokens).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

# Make build_index.py importable so we reuse its exact constants/functions
DATA_SCRIPTS_DIR = Path(r"C:\Programming\rag_seminar\data-science-seminar\src\data-science-seminar\data_scripts")
sys.path.insert(0, str(DATA_SCRIPTS_DIR))

from build_index import MODEL_NAME, MAX_SEQ_LENGTH, DEVICE, format_date  # noqa: E402

print(f"MODEL_NAME     = {MODEL_NAME!r}")
print(f"MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH!r}")
print(f"DEVICE         = {DEVICE!r}")


C:\Programming\rag_seminar\data-science-seminar\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MODEL_NAME     = 'multi-qa-mpnet-base-dot-v1'
MAX_SEQ_LENGTH = 512
DEVICE         = 'cpu'


## Load model (identical to build_index.py)

In [2]:
# Same construction as in build_index.main():
#   model = SentenceTransformer(MODEL_NAME, device=DEVICE)
#   model.max_seq_length = MAX_SEQ_LENGTH
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = MAX_SEQ_LENGTH

print(f"Tokenizer class: {type(model.tokenizer).__name__}")
print(f"model.max_seq_length: {model.max_seq_length}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8575.37it/s]

Tokenizer class: MPNetTokenizer
model.max_seq_length: 512


## Test dates

Formatting the dates the same way as in the date injection

In [3]:
TEST_DATES_ISO = [
    "2024-08-12",  # zweistelliger Tag
    "2017-01-01",  # einstelliger Tag, Jahresanfang
    "2024-09-05",  # einstelliger Tag
    "2019-12-31",  # zweistelliger Tag, Monatsende (31 Tage)
    "2020-02-29",  # Schaltjahr, 29. Februar
    "2018-04-30",  # Monat mit 30 Tagen
    "2022-03-03",  # einstelliger Tag
    "2016-06-06",  # einstelliger Tag
    "2023-10-10",  # zweistelliger Tag
    "2025-07-04",  # einstelliger Tag
]

# INJECTION_WEIGHT=1 reproduces build_passage()'s date-injection snippet
# exactly: 'Published on {formatted_date}. ' repeated `injection_weight` times.
INJECTION_WEIGHT = 1


def build_injected_snippet(formatted_date: str) -> str:
    date = ""
    for _ in range(INJECTION_WEIGHT):
        date += f"Published on {formatted_date}. "
    return date


for iso in TEST_DATES_ISO:
    print(f"{iso}  ->  {format_date(iso)!r}")


2024-08-12  ->  'August 12, 2024'
2017-01-01  ->  'January 1, 2017'
2024-09-05  ->  'September 5, 2024'
2019-12-31  ->  'December 31, 2019'
2020-02-29  ->  'February 29, 2020'
2018-04-30  ->  'April 30, 2018'
2022-03-03  ->  'March 3, 2022'
2016-06-06  ->  'June 6, 2016'
2023-10-10  ->  'October 10, 2023'
2025-07-04  ->  'July 4, 2025'


## Tokenization (exactly like in build_index.py)


In [4]:
rows = []
for iso in TEST_DATES_ISO:
    formatted = format_date(iso)
    snippet = build_injected_snippet(formatted)

    # add_special_tokens=False: in the real pipeline the date/snippet is
    # injected into the middle of a passage, so <s>/</s> are only added once
    # for the whole passage, not per injected snippet. Tokenizing without
    # special tokens gives the actual marginal token cost of the injection.
    date_ids = model.tokenizer(formatted, add_special_tokens=False)["input_ids"]
    snippet_ids = model.tokenizer(snippet, add_special_tokens=False)["input_ids"]

    date_tokens = model.tokenizer.convert_ids_to_tokens(date_ids)
    snippet_tokens = model.tokenizer.convert_ids_to_tokens(snippet_ids)

    rows.append({
        "iso_date": iso,
        "formatted_date": formatted,
        "date_token_count": len(date_ids),
        "date_input_ids": date_ids,
        "date_tokens": date_tokens,
        "snippet": snippet,
        "snippet_token_count": len(snippet_ids),
        "snippet_input_ids": snippet_ids,
        "snippet_tokens": snippet_tokens,
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
print(f"Tokenized {len(df)} dates.")


Tokenized 10 dates.


## Token count per date

In [5]:
summary = df[["iso_date", "formatted_date", "date_token_count", "snippet_token_count"]]
summary


,iso_date,formatted_date,date_token_count,snippet_token_count
0,2024-08-12,"August 12, 2024",5,8
1,2017-01-01,"January 1, 2017",4,7
2,2024-09-05,"September 5, 2024",5,8
3,2019-12-31,"December 31, 2019",4,7
4,2020-02-29,"February 29, 2020",4,7
5,2018-04-30,"April 30, 2018",4,7
6,2022-03-03,"March 3, 2022",5,8
7,2016-06-06,"June 6, 2016",4,7
8,2023-10-10,"October 10, 2023",5,8
9,2025-07-04,"July 4, 2025",5,8


## Detail output per date (individual tokens)

Why do some dates produce 5 instead of 4 tokens? --> year is sometimes split into 2 tokens.

Note: `<s>`/`</s>` are excluded here (`add_special_tokens=False`) since under realistic conditions the date snippet is injected into the middle of an existing passage, so these tokens are already present regardless of injection and shouldn't be double-counted.

In [6]:
for row in rows:
    print(f"\n{row['formatted_date']!r}  ({row['date_token_count']} tokens)")
    print(f"  tokens   : {row['date_tokens']}")
    print(f"Snippet tokens: {row['snippet_tokens']}")



'August 12, 2024'  (5 tokens)
  tokens   : ['august', '12', ',', '202', '##4']
Snippet tokens: ['published', 'on', 'august', '12', ',', '202', '##4', '.']

'January 1, 2017'  (4 tokens)
  tokens   : ['january', '1', ',', '2017']
Snippet tokens: ['published', 'on', 'january', '1', ',', '2017', '.']

'September 5, 2024'  (5 tokens)
  tokens   : ['september', '5', ',', '202', '##4']
Snippet tokens: ['published', 'on', 'september', '5', ',', '202', '##4', '.']

'December 31, 2019'  (4 tokens)
  tokens   : ['december', '31', ',', '2019']
Snippet tokens: ['published', 'on', 'december', '31', ',', '2019', '.']

'February 29, 2020'  (4 tokens)
  tokens   : ['february', '29', ',', '2020']
Snippet tokens: ['published', 'on', 'february', '29', ',', '2020', '.']

'April 30, 2018'  (4 tokens)
  tokens   : ['april', '30', ',', '2018']
Snippet tokens: ['published', 'on', 'april', '30', ',', '2018', '.']

'March 3, 2022'  (5 tokens)
  tokens   : ['march', '3', ',', '202', '##2']
Snippet tokens: ['pub